# CALCE Battery Data — Direct Ingestion Notebook

**Source:** Center for Advanced Life Cycle Engineering, University of Maryland  
**Cycler:** Arbin BT2000  
**Format:** Arbin Excel export (.xlsx) — ingested directly via psycopg2

### How to add a new dataset
1. Download the `.xlsx` file from calce.umd.edu/battery-data
2. Add a new entry to the `DATASETS` list in Section 1
3. Run the notebook top to bottom

### Column mapping
| Arbin column | battdb column | Conversion |
|---|---|---|
| `Voltage(V)` | `voltage_mv` | ×1000 |
| `Current(A)` | `current_ma` | ×1000 |
| `Charge_Capacity(Ah)` | `charge_capacity_mah` | ×1000 |
| `Discharge_Capacity(Ah)` | `discharge_capacity_mah` | ×1000 |
| `Charge_Energy(Wh)` | `charge_energy_mwh` | ×1000 |
| `Discharge_Energy(Wh)` | `discharge_energy_mwh` | ×1000 |
| `Test_Time(s)` | `test_time_s` | — |
| `Date_Time` | `date_time` | — |
| `Step_Index` | `step_index` | — |
| `Cycle_Index` | `cycle_index` | — |
| `Temperature(C)_1/2`, `Internal_Resistance`, etc. | `other_details` | jsonb |

### Prerequisites
- battdb running on `localhost:5454` (`docker compose up -d`)
- `.env` file with DB credentials

## 1. Imports & dataset registry

In [ ]:
import os
import json
import warnings
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (12, 4), 'axes.grid': True, 'grid.alpha': 0.3})

ENV_PATH = r"C:\Users\Vcpat\Downloads\battetl-main\battetl-main\.env"
DATA_DIR = r"C:\Users\Vcpat\Downloads\A123_OCV-10-20120629\OCV-10-20120629"

load_dotenv(ENV_PATH, override=True)

# ── Dataset registry ────────────────────────────────────────────────────────
# Add one entry per cell/test file.

DATASETS = [
    {
        "file":              "A1-007-OCV-10-20120629.xlsx",
        "test_name":         "CALCE_A1-007_OCV_neg10C_20120629",
        "channel":           1,
        "comments":          "OCV test at -10C. CALCE A1 series LCO cell.",
        "manufacturer_sn":   "A1-007",
        "batch_number":      "A1",
        "manufacturer":      "Unknown",
        "manufacturer_pn":   "CALCE-A1",
        "form_factor":       "prismatic",
        "chemistry":         "LCO",
        "cycler_sn":         "CALCE-ARBIN-CH005",
        "cycler_model":      "BT2000",
        "schedule_name":     "OCV-10-20120629",
        "test_type":         "Characterization",
        "project":           "CALCE_A1_OCV",
    },
    {
        "file":              "A1-008-OCV-10-20120629.xlsx",
        "test_name":         "CALCE_A1-008_OCV_neg10C_20120629",
        "channel":           1,
        "comments":          "OCV test at -10C. CALCE A1 series LCO cell.",
        "manufacturer_sn":   "A1-008",
        "batch_number":      "A1",
        "manufacturer":      "Unknown",
        "manufacturer_pn":   "CALCE-A1",
        "form_factor":       "prismatic",
        "chemistry":         "LCO",
        "cycler_sn":         "CALCE-ARBIN-CH006",
        "cycler_model":      "BT2000",
        "schedule_name":     "OCV-10-20120629",
        "test_type":         "Characterization",
        "project":           "CALCE_A1_OCV",
    },
    # ── Add more datasets here ──────────────────────────────────────────────
    # {
    #     "file":            "CS2-33-...-20130101.xlsx",
    #     "test_name":       "CALCE_CS2-33_cycle_25C",
    #     ...etc
    # },
]

print(f"Registered datasets: {len(DATASETS)}")
for d in DATASETS:
    path = os.path.join(DATA_DIR, d['file'])
    print(f"  {'OK     ' if os.path.exists(path) else 'MISSING'} {d['file']}")

## 2. Inspect files

In [ ]:
def get_data_sheet(path):
    xl = pd.ExcelFile(path)
    return [s for s in xl.sheet_names if s not in ('Info', 'Sheet1')][0]

for d in DATASETS:
    path = os.path.join(DATA_DIR, d['file'])
    if not os.path.exists(path):
        print(f"MISSING: {d['file']}")
        continue
    sheet = get_data_sheet(path)
    df = pd.read_excel(path, sheet_name=sheet)
    print(f"=== {d['file']} ===")
    print(f"  Sheet:       {sheet}")
    print(f"  Rows:        {len(df):,}")
    print(f"  Date range:  {df['Date_Time'].min()} -> {df['Date_Time'].max()}")
    print(f"  Cycles:      {df['Cycle_Index'].nunique()}")
    print(f"  Voltage:     {df['Voltage(V)'].min():.3f} - {df['Voltage(V)'].max():.3f} V")
    print(f"  Current:     {df['Current(A)'].min():.3f} - {df['Current(A)'].max():.3f} A")
    print()

## 3. Plot raw data

In [ ]:
for d in DATASETS:
    path = os.path.join(DATA_DIR, d['file'])
    if not os.path.exists(path):
        continue
    sheet = get_data_sheet(path)
    df = pd.read_excel(path, sheet_name=sheet)
    t = df['Test_Time(s)'] / 3600

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(d['test_name'], fontsize=12)

    axes[0].plot(t, df['Voltage(V)'], color='steelblue', linewidth=0.6)
    axes[0].set_xlabel('Time (h)')
    axes[0].set_ylabel('Voltage (V)')
    axes[0].set_title('Voltage')

    axes[1].plot(t, df['Current(A)'], color='coral', linewidth=0.6)
    axes[1].set_xlabel('Time (h)')
    axes[1].set_ylabel('Current (A)')
    axes[1].set_title('Current')

    axes[2].plot(t, df['Temperature (C)_1'], color='seagreen', linewidth=0.6, label='T1')
    axes[2].plot(t, df['Temperature (C)_2'], color='mediumorchid', linewidth=0.6, label='T2')
    axes[2].set_xlabel('Time (h)')
    axes[2].set_ylabel('Temperature (C)')
    axes[2].set_title('Temperature')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## 4. Connect to battdb

In [ ]:
conn = psycopg2.connect(
    host='localhost', port=5454,
    dbname='battdb', user='postgres', password='password'
)
print('Connected to battdb')

# Check test_data columns
cur = conn.cursor()
cur.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = 'test_data'
    ORDER BY ordinal_position
""")
print('\ntest_data columns:')
for col, dtype in cur.fetchall():
    print(f'  {col}: {dtype}')
cur.close()

## 5. Insertion function

Maps Arbin Excel columns directly to battdb schema.

In [ ]:
def insert_arbin_direct(conn, d, data_dir):
    cur = conn.cursor()

    # ── Read file ──────────────────────────────────────────────────────────
    path = os.path.join(data_dir, d['file'])
    sheet = get_data_sheet(path)
    df = pd.read_excel(path, sheet_name=sheet)
    print(f"  Read {len(df):,} rows from {d['file']}")

    # ── 1. Get or create cells_meta ────────────────────────────────────────
    cur.execute("SELECT cell_type_id FROM cells_meta WHERE manufacturer_pn = %s",
                (d['manufacturer_pn'],))
    result = cur.fetchone()
    if not result:
        cur.execute("""
            INSERT INTO cells_meta (manufacturer, manufacturer_pn, form_factor, chemistry)
            VALUES (%s, %s, %s, %s) RETURNING cell_type_id
        """, (d['manufacturer'], d['manufacturer_pn'], d['form_factor'], d['chemistry']))
        result = cur.fetchone()
    cell_type_id = result[0]

    # ── 2. Get or create cell ──────────────────────────────────────────────
    cur.execute("SELECT cell_id FROM cells WHERE manufacturer_sn = %s",
                (d['manufacturer_sn'],))
    result = cur.fetchone()
    if not result:
        cur.execute("""
            INSERT INTO cells (manufacturer_sn, cell_type_id, batch_number)
            VALUES (%s, %s, %s) RETURNING cell_id
        """, (d['manufacturer_sn'], cell_type_id, d['batch_number']))
        result = cur.fetchone()
    cell_id = result[0]

    # ── 3. Get or create cyclers_meta ──────────────────────────────────────
    cur.execute("SELECT cycler_type_id FROM cyclers_meta WHERE model = %s",
                (d['cycler_model'],))
    result = cur.fetchone()
    if not result:
        cur.execute("""
            INSERT INTO cyclers_meta (manufacturer, model)
            VALUES (%s, %s) RETURNING cycler_type_id
        """, ('Arbin', d['cycler_model']))
        result = cur.fetchone()
    cycler_type_id = result[0]

    # ── 4. Get or create cycler ────────────────────────────────────────────
    cur.execute("SELECT cycler_id FROM cyclers WHERE sn = %s", (d['cycler_sn'],))
    result = cur.fetchone()
    if not result:
        cur.execute("""
            INSERT INTO cyclers (sn, cycler_type_id, location)
            VALUES (%s, %s, %s) RETURNING cycler_id
        """, (d['cycler_sn'], cycler_type_id, 'University of Maryland'))
        result = cur.fetchone()
    cycler_id = result[0]

    # ── 5. Get or create schedule_meta ─────────────────────────────────────
    cur.execute("SELECT schedule_id FROM schedule_meta WHERE schedule_name = %s",
                (d['schedule_name'],))
    result = cur.fetchone()
    if not result:
        cur.execute("""
            INSERT INTO schedule_meta (schedule_name, cycler_make, test_type)
            VALUES (%s, %s, %s) RETURNING schedule_id
        """, (d['schedule_name'], 'Arbin', d['test_type']))
        result = cur.fetchone()
    schedule_id = result[0]

    # ── 6. Get or create test_meta ─────────────────────────────────────────
    cur.execute("SELECT test_id FROM test_meta WHERE test_name = %s", (d['test_name'],))
    result = cur.fetchone()
    if not result:
        start_dt = df['Date_Time'].min()
        end_dt   = df['Date_Time'].max()
        cur.execute("""
            INSERT INTO test_meta (
                test_name, cell_id, channel, comments,
                schedule_id, cycler_id,
                first_recorded_datetime, last_recorded_datetime
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            RETURNING test_id
        """, (
            d['test_name'], cell_id, d['channel'], d['comments'],
            schedule_id, cycler_id,
            start_dt, end_dt
        ))
        result = cur.fetchone()
    test_id = result[0]

    # ── 7. Insert test_data rows ───────────────────────────────────────────
    rows = [
        (
            test_id,
            int(r['Data_Point']),
            round(r['Test_Time(s)'], 3),
            r['Date_Time'].isoformat() if pd.notna(r['Date_Time']) else None,
            round(r['Step_Time(s)'], 3),
            int(r['Step_Index']),
            int(r['Cycle_Index']),
            round(r['Current(A)'] * 1000, 4),
            round(r['Voltage(V)'] * 1000, 4),
            round(r['Charge_Capacity(Ah)'] * 1000, 6),
            round(r['Discharge_Capacity(Ah)'] * 1000, 6),
            round(r['Charge_Energy(Wh)'] * 1000, 6),
            round(r['Discharge_Energy(Wh)'] * 1000, 6),
            json.dumps({
                'dVdt':           round(float(r['dV/dt(V/s)']), 8),
                'IR_ohm':         round(float(r['Internal_Resistance(Ohm)']), 6),
                'AC_impedance':   round(float(r['AC_Impedance(Ohm)']), 6),
                'ACI_phase_deg':  round(float(r['ACI_Phase_Angle(Deg)']), 4),
                'temperature_1_c': round(float(r['Temperature (C)_1']), 4),
                'temperature_2_c': round(float(r['Temperature (C)_2']), 4),
            })
        )
        for _, r in df.iterrows()
    ]

    cur.executemany("""
        INSERT INTO test_data (
            test_id, data_point, test_time_s, date_time,
            step_time_s, step_index, cycle_index,
            current_ma, voltage_mv,
            charge_capacity_mah, discharge_capacity_mah,
            charge_energy_mwh, discharge_energy_mwh,
            other_details
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING
    """, rows)

    conn.commit()
    cur.close()
    return test_id, len(rows)

print('Function defined.')

## 6. Test run — single cell

In [ ]:
test_id, n = insert_arbin_direct(conn, DATASETS[0], DATA_DIR)
print(f"test_id={test_id}, {n:,} rows inserted")

## 7. Ingest all datasets

In [ ]:
from tqdm.notebook import tqdm

results = {'success': [], 'failed': []}

for d in tqdm(DATASETS, desc='Ingesting'):
    if not os.path.exists(os.path.join(DATA_DIR, d['file'])):
        print(f"  SKIP {d['test_name']} — file not found")
        continue
    try:
        test_id, n = insert_arbin_direct(conn, d, DATA_DIR)
        results['success'].append(d['test_name'])
        print(f"  OK  {d['test_name']}: {n:,} rows")
    except Exception as e:
        conn.rollback()
        results['failed'].append((d['test_name'], str(e)))
        print(f"  FAIL {d['test_name']}: {e}")

print(f"\nSuccess: {len(results['success'])}")
print(f"Failed:  {len(results['failed'])}")

## 8. Verify — query battdb

In [ ]:
cur = conn.cursor()
cur.execute("""
    SELECT tm.test_name, COUNT(td.*) as rows,
           MIN(td.date_time) as start,
           MAX(td.date_time) as end
    FROM test_meta tm
    JOIN test_data td ON tm.test_id = td.test_id
    WHERE tm.test_name LIKE 'CALCE_%'
    GROUP BY tm.test_name
    ORDER BY tm.test_name
""")
rows = cur.fetchall()
cur.close()

print(f"{'Test name':<45} {'Rows':>8}  Start")
print('-' * 80)
for r in rows:
    print(f"{r[0]:<45} {r[1]:>8,}  {r[2]}")

In [ ]:
# Plot voltage and current from battdb
query = """
    SELECT td.test_time_s, td.voltage_mv, td.current_ma,
           td.charge_capacity_mah, td.discharge_capacity_mah,
           td.other_details
    FROM test_data td
    JOIN test_meta tm ON td.test_id = tm.test_id
    WHERE tm.test_name = %s
    ORDER BY td.test_time_s
"""
db_df = pd.read_sql(query, conn, params=(DATASETS[0]['test_name'],))
print(f"Rows from battdb: {len(db_df):,}")

t = db_df['test_time_s'] / 3600

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f"{DATASETS[0]['test_name']} — from battdb", fontsize=12)

axes[0,0].plot(t, db_df['voltage_mv'] / 1000, color='steelblue', linewidth=0.6)
axes[0,0].set_ylabel('Voltage (V)')
axes[0,0].set_title('Voltage')

axes[0,1].plot(t, db_df['current_ma'] / 1000, color='coral', linewidth=0.6)
axes[0,1].set_ylabel('Current (A)')
axes[0,1].set_title('Current')

axes[1,0].plot(t, db_df['charge_capacity_mah'], color='seagreen', linewidth=0.6, label='Charge')
axes[1,0].plot(t, db_df['discharge_capacity_mah'], color='tomato', linewidth=0.6, label='Discharge')
axes[1,0].set_xlabel('Time (h)')
axes[1,0].set_ylabel('Capacity (mAh)')
axes[1,0].set_title('Capacity')
axes[1,0].legend()

# Extract temperature from other_details jsonb
db_df['T1'] = db_df['other_details'].apply(
    lambda x: (x if isinstance(x, dict) else json.loads(x)).get('temperature_1_c', None)
)
axes[1,1].plot(t, db_df['T1'], color='purple', linewidth=0.6)
axes[1,1].set_xlabel('Time (h)')
axes[1,1].set_ylabel('Temperature (C)')
axes[1,1].set_title('Temperature (channel 1)')

for ax in axes.flat:
    if ax.get_xlabel() == '':
        ax.set_xlabel('Time (h)')

plt.tight_layout()
plt.show()

conn.close()
print('Done.')